# Train Eval Experiment 

## Resources

### Import Library

In [5]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# import SKlearn FFNN
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# import custom module
from ffnn import FFNN, Layer

#### Import Dataset

In [6]:
X_train = np.load('dataset/X_train_final.npy')
y_train = np.load('dataset/y_train_final.npy')
X_test = np.load('dataset/X_test_final.npy')
y_test = np.load('dataset/y_test_final.npy')

# Cek shape data
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (9844, 21)
y_train shape: (9844,)
X_test shape: (2000, 21)
y_test shape: (2000,)


## Experiment

### Architecture 1

Input (21)
Hidden 1 (32)
Hidden 2 (16)
Output (1)

#### SKLearn

In [7]:
# activations test
sklearn_activations = ['identity', 'relu', 'logistic', 'tanh']
sklearn_models = {}

for activation in sklearn_activations:
    mlp = MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation=activation,   
        solver='sgd',
        learning_rate_init=0.01,
        max_iter=50,
        batch_size=32,
        random_state=42
    )
    mlp.fit(X_train, y_train)
    sklearn_models[activation] = mlp
    print(f"Finish model {activation}")

Finish model identity


/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Finish model relu


/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Finish model logistic
Finish model tanh


/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


In [8]:
# Evaluation
for activation, model in sklearn_models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy for {activation}: {acc:.4f}")
    print(f"Classification Report for {activation}:\n{classification_report(y_test, y_pred)}")
    print(f"Confusion Matrix for {activation}:\n{confusion_matrix(y_test, y_pred)}\n")
    

Accuracy for identity: 0.7355
Classification Report for identity:
              precision    recall  f1-score   support

           0       0.65      0.66      0.66       769
           1       0.79      0.78      0.78      1231

    accuracy                           0.74      2000
   macro avg       0.72      0.72      0.72      2000
weighted avg       0.74      0.74      0.74      2000

Confusion Matrix for identity:
[[510 259]
 [270 961]]

Accuracy for relu: 0.7165
Classification Report for relu:
              precision    recall  f1-score   support

           0       0.64      0.60      0.62       769
           1       0.76      0.79      0.77      1231

    accuracy                           0.72      2000
   macro avg       0.70      0.70      0.70      2000
weighted avg       0.71      0.72      0.71      2000

Confusion Matrix for relu:
[[465 304]
 [263 968]]

Accuracy for logistic: 0.7345
Classification Report for logistic:
              precision    recall  f1-score   supp

#### Custom FFNN

In [9]:
model = FFNN(loss='bce')
model.add(Layer(input_size=21, output_size=32, activation='relu', init_method='he'))
model.add(Layer(input_size=32, output_size=16, activation='relu', init_method='he'))
model.add(Layer(input_size=16, output_size=1, activation='sigmoid', init_method='xavier'))

training = model.fit(
    X_train,
    y_train.reshape(-1,1),
    epochs=50,
    batch_size=32, 
    learning_rate=0.01, 
    optimizer='adam',
    validation_data=(X_test, y_test.reshape(-1, 1))
)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

epoch 1/50 - loss: 0.4877 - val_loss: 0.5285
epoch 2/50 - loss: 0.4368 - val_loss: 0.5162
epoch 3/50 - loss: 0.4245 - val_loss: 0.5201
epoch 4/50 - loss: 0.4181 - val_loss: 0.5197
epoch 5/50 - loss: 0.4172 - val_loss: 0.5198
epoch 6/50 - loss: 0.4143 - val_loss: 0.5141
epoch 7/50 - loss: 0.4099 - val_loss: 0.5222
epoch 8/50 - loss: 0.4092 - val_loss: 0.5326
epoch 9/50 - loss: 0.4078 - val_loss: 0.5227
epoch 10/50 - loss: 0.4046 - val_loss: 0.5190
epoch 11/50 - loss: 0.4024 - val_loss: 0.5173
epoch 12/50 - loss: 0.4002 - val_loss: 0.5204
epoch 13/50 - loss: 0.3967 - val_loss: 0.5306
epoch 14/50 - loss: 0.3959 - val_loss: 0.5256
epoch 15/50 - loss: 0.3949 - val_loss: 0.5286
epoch 16/50 - loss: 0.3937 - val_loss: 0.5382
epoch 17/50 - loss: 0.3923 - val_loss: 0.5302
epoch 18/50 - loss: 0.3908 - val_loss: 0.5460
epoch 19/50 - loss: 0.3880 - val_loss: 0.5291
epoch 20/50 - loss: 0.3870 - val_loss: 0.5412
epoch 21/50 - loss: 0.3847 - val_loss: 0.5535
epoch 22/50 - loss: 0.3804 - val_loss: 0.54